# Phase 5 - Stage 2 span extraction (Kaggle or Colab)

Steps 5.4-5.6 of `PLAN.md`: runs 9-11 of the experiment matrix.

| Run | Model | Varies | Step |
|---|---|---|---|
| 9 | BiLSTM + softmax | per-token decisions | 5.4 |
| 10 | BiLSTM + **CRF** | Viterbi over learned transitions | 5.5 |
| 11 | BiomedBERT token-cls | contextual | 5.6 |

Steps 5.1-5.3 are already done: the BIO converter, its unit tests and the
20-example spot-check are committed. Steps 5.7-5.9 (strict/lenient scoring,
the illegal-sequence count, the published-corpus check) run **locally** after
this notebook, from the prediction files it saves.

**Accelerator: GPU T4 x2** (cell 1 pins one card). **Internet: On** for run
11's checkpoint download.

### What makes runs 9 and 10 a valid ablation

They must differ *only* in `--crf`. Same embedding matrix (E3, the Phase 4
winner), same seed, same hyperparameters, same split, same device count. The
training script hard-fails if two GPUs are visible, because `DataParallel`
changes both the effective batch and how the CRF loss is reduced -
`pytorch-crf` computes its loss inside `forward`, so two devices return a
loss vector rather than a scalar (`PLAN.md` F8).


## 1. Environment, and the single-GPU pin (PLAN F8)

**First cell, before `import torch`.** `CUDA_VISIBLE_DEVICES` is read once,
when the CUDA context initialises; setting it after anything has touched
`torch.cuda` is silently ignored for the rest of the kernel's life.

`pytorch-crf` is the one install this notebook needs - it is not on the
Kaggle base image. It is pinned, like everything else in GROUP A of
`requirements-remote.txt`.


In [ ]:
import os

# BEFORE importing torch. Not stylistic - see the note above.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import sys, subprocess, pathlib, time

ON_KAGGLE = pathlib.Path('/kaggle').exists()
ON_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').exists()
PLATFORM = 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local'
WORK = pathlib.Path('/kaggle/working' if ON_KAGGLE else
                    '/content' if ON_COLAB else '.')
print('platform :', PLATFORM)

!pip install -q pytorch-crf==0.7.2 seqeval==1.2.2

import torch, transformers, datasets, numpy, seqeval, torchcrf
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('seqeval', seqeval.__version__, '| pytorch-crf installed')

n = torch.cuda.device_count()
print('CUDA devices visible:', n)
print('GPU:', torch.cuda.get_device_name(0) if n else 'CPU')
assert n <= 1, (
    f'{n} GPUs visible - the pin did not take effect. Something touched torch.cuda '
    'before this cell. Fix: Run -> Restart & clear cell outputs, then run THIS cell first.')

print(f'\nRECORD: torch={torch.__version__} transformers={transformers.__version__} devices={n}')


## 2. Get the code

The frozen splits are committed, so the clone brings the Stage 2 spans with
it. The BIO tags are **not** committed - they are derived from those spans by
`src/bio_convert.py` at the start of every run, which is what stops the
converter and the training data drifting apart.


In [ ]:
REPO_URL = 'https://github.com/sifatul-islam-onik/ADE-Sentinel.git'
REPO = WORK / 'ADE-Sentinel'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)

sys.path.insert(0, str(REPO))
os.chdir(REPO)
GIT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('repo at', GIT_COMMIT)

import pandas as pd
for split in ('train', 'dev', 'test'):
    df = pd.read_parquet(REPO / 'data' / 'splits' / f'stage2_{split}.parquet')
    print(f'  stage2_{split}: {len(df):,} sentences, {df.n_spans.sum():,} spans')


## 3. Test before training

The tagger tests are skipped locally (no torch) and the CRF tests need
`pytorch-crf`, so this is the first machine that can run either.

The one that matters most is `test_crf_never_emits_an_illegal_transition`.
That is run 10's entire claim, and it is a *structural* property of Viterbi
decoding - so it must hold even at random initialisation. If it fails here,
any illegal-sequence count you report afterwards is measuring a bug.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_bilstm_tagger.py', 'tests/test_stage2_metrics.py',
                'tests/test_bio_convert.py'], check=True)


## 4. Embedding matrices

Same Kaggle Dataset as Phase 4 - the `.npy` files are gitignored (`PLAN.md`
F7), so cloning does not bring them. Attach the dataset via **Add Input** and
put its version in `DATASET_VERSION`.

Runs 9-10 use **E3 (FastText)**, which won the Phase 4 ablation at macro-F1
0.8785 against 0.7942 for GloVe. `PLAN.md` 5.4 says "best of E1-E3"; that is
what the four-bar chart selected.


In [ ]:
DATASET_VERSION = ''   # <- the version shown in the Input panel

import json, numpy as np


def holds_matrices(d):
    return d.is_dir() and (d / 'vocab.json').exists() and any(d.glob('E*.npy'))


candidates = []
if ON_KAGGLE:
    root = pathlib.Path('/kaggle/input')
    candidates += [p.parent for p in root.rglob('vocab.json')]
    candidates += [root] + sorted(root.glob('*')) + sorted(root.glob('*/*'))
candidates.append(REPO / 'models' / 'emb_matrices')

MATRICES = next((d for d in candidates if holds_matrices(d)), None)
if MATRICES is None:
    listing = []
    if ON_KAGGLE and pathlib.Path('/kaggle/input').exists():
        for p in sorted(pathlib.Path('/kaggle/input').rglob('*'))[:40]:
            listing.append(f'    {p}')
    raise SystemExit(
        'No directory with both vocab.json and E*.npy was found.\n\n'
        'Mounted under /kaggle/input:\n'
        + ('\n'.join(listing) if listing else '    (nothing attached)')
        + '\n\nAttach the ade-sentinel-artifacts Dataset via Add Input.')

print('matrices :', MATRICES)
print('E3       :', np.load(MATRICES / 'E3.npy', mmap_mode='r').shape)
if not DATASET_VERSION:
    print('\nWARNING: DATASET_VERSION is blank - the run rows lose their input provenance (PLAN F10)')


## 5. Runs 9 and 10 - the CRF ablation

One command each. **`--crf` is the only difference.**

Watch the `illegal` column as the epochs print: run 9 will show a non-zero
count that fluctuates, and run 10 should sit at exactly zero from the first
epoch. That contrast is step 5.8, and it is visible during training rather
than only at the end.

A few minutes each on one T4.


In [ ]:
def train_tagger(run_id, crf=False, **kw):
    cmd = [sys.executable, 'scripts/train_bilstm_tagger.py',
           '--run-id', str(run_id), '--embedding', 'E3',
           '--matrices', str(MATRICES), '--dataset-version', DATASET_VERSION]
    if crf:
        cmd.append('--crf')
    for k, v in kw.items():
        cmd += [f'--{k.replace("_", "-")}', str(v)]
    print('\n' + '=' * 70)
    print(' '.join(cmd[1:]))
    print('=' * 70)
    subprocess.run(cmd, check=True)


t0 = time.time()
train_tagger('9', crf=False)
train_tagger('10', crf=True)
print(f'\nruns 9-10 complete in {(time.time() - t0) / 60:.1f} min')


## 6. Run 11 - BERT token classification

BiomedBERT, the Phase 4 transformer winner (0.9402 against 0.9159 for
bert-base-uncased).

**The subword alignment is the whole risk here.** BIO tags are one per word;
WordPiece splits words further. The script labels the first subword of each
word, masks the rest with -100, and decodes predictions back to word level
before scoring - so run 11's entity counts are comparable with runs 9-10.
Scoring at subword level would make the same entity count for a different
number of tokens, and the three runs would no longer be on one axis.

Roughly 5-10 minutes on a T4.


In [ ]:
cmd = [sys.executable, 'scripts/train_bert_tagger.py',
       '--run-id', '11', '--model', 'biomedbert',
       '--dataset-version', DATASET_VERSION]
print(' '.join(cmd[1:]))
t0 = time.time()
subprocess.run(cmd, check=True)
print(f'\nrun 11 complete in {(time.time() - t0) / 60:.1f} min')


## 7. Save the outputs - before the session ends

| Artefact | Where it goes |
|---|---|
| `runs.csv` rows | **commit to git** |
| `test_predictions.json` (all three runs) | **bring home** - steps 5.7-5.9 rescore from these locally, and Phase 6 needs them |
| checkpoints | Kaggle Dataset; the BiLSTM-CRF one also ships in the Phase 7 demo |

The prediction files are the important ones and they are small - a few
hundred KB of tag strings. Everything in the report after this point is
computed from them on the local machine, which is what makes the Stage 2
numbers reproducible without a GPU.


In [ ]:
import shutil

bundle = WORK / 'phase5_outputs'
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

shutil.copy2('results/runs.csv', bundle / 'runs.csv')
if pathlib.Path('models/stage2').exists():
    shutil.copytree('models/stage2', bundle / 'stage2_models',
                    ignore=shutil.ignore_patterns('trainer'))

total = sum(f.stat().st_size for f in bundle.rglob('*') if f.is_file())
print(f'{bundle}  ({total / 1e6:,.0f} MB)')
for f in sorted(bundle.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(bundle)}  {f.stat().st_size / 1e6:,.2f} MB')


---

## Then, locally - steps 5.7-5.9

```bash
.venv\Scripts\python scripts\stage2_report.py
```

That rescores every run from the saved predictions, writes
`results/figures/stage2_results.md` and the two-panel CRF figure, and checks
the BIO conversion against the published corpus statistics. It also
recomputes each logged entity-F1 independently and flags any disagreement -
the remote session is trusted for the training, not for the numbers.

## Phase 5 exit criterion

From `PLAN.md`: the entity-F1 table is complete and the CRF illegal-sequence
count is obtained.

Read the result honestly:

- **The CRF should drive illegal sequences to zero.** That is structural, not
  a training outcome. If it does not, the decode is wrong - diagnose before
  reporting.
- **Entity-F1 may barely move between runs 9 and 10.** That is the expected
  result and it is not a disappointment. Report both axes: on F1 the CRF looks
  like a no-op, and the illegal-sequence count is where it earns its place.
  A tagger whose output is always well-formed is the one you can put behind
  the Phase 7 demo with no repair step.
- **Run 11 should lead on F1.** If it does not, check the subword alignment
  before concluding anything about transformers.

Then Phase 6: run 12 chains the best Stage 1 model (run 8, BiomedBERT) into
the best Stage 2 model on the shared test split - valid only because of the
single global split (`PLAN.md` F4).
